In [1]:
import os
import cv2
import numpy as np
import pandas as pd
from skimage.feature import hog, graycomatrix, graycoprops

def Extract_Advanced_Features(img_path):
    # 1. Đọc ảnh
    img = cv2.imread(img_path)
    if img is None:
        print(f"Lỗi đọc ảnh: {img_path}")
        return None
    
    # Đưa ảnh về kích thước chuẩn 128x128 để thuật toán HOG chạy ổn định
    img = cv2.resize(img, (128, 128))
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # 2. Đặc trưng Màu sắc (Color - RGB Mean)
    # Lấy giá trị trung bình của 3 kênh màu B, G, R
    avg_color = np.mean(img, axis=(0, 1)) 
    
    # 3. Đặc trưng Kết cấu (Texture - GLCM)
    # Tính toán độ tương phản (contrast) và sự tương quan (correlation) của bề mặt
    glcm = graycomatrix(gray, distances=[1], angles=[0], levels=256, symmetric=True, normed=True)
    contrast = graycoprops(glcm, 'contrast')[0, 0]
    correlation = graycoprops(glcm, 'correlation')[0, 0]
    
    # 4. Đặc trưng Hình dáng (HOG - Histogram of Oriented Gradients)
    # Bắt các đặc điểm góc cạnh như tai mèo, mõm chó
    hog_features = hog(gray, orientations=9, pixels_per_cell=(8, 8),
                       cells_per_block=(2, 2), block_norm='L2-Hys', visualize=False)
    
    # 5. Gộp tất cả lại thành 1 mảng 1 chiều (Vector)
    combined_features = np.hstack((avg_color, contrast, correlation, hog_features))
    return combined_features

def Create_DogCat_Dataset():
    # File code nằm trong src, nên đường dẫn phải lùi ra 1 bước (../) để vào data
    base_dir = '../data/train'
    categories = ['dog', 'cat']
    dataset = []
    
    for category in categories:
        folder_path = os.path.join(base_dir, category)
        
        # Quét qua từng file ảnh trong thư mục
        if os.path.exists(folder_path):
            for file_name in os.listdir(folder_path):
                img_path = os.path.join(folder_path, file_name)
                features = Extract_Advanced_Features(img_path)
                
                if features is not None:
                    # Chuyển Numpy array thành Python list và gắn nhãn (Label) ở cột cuối
                    row = features.tolist() + [category]
                    dataset.append(row)
                    print(f"Đã xử lý xong: {img_path}")
        else:
            print(f"Không tìm thấy thư mục: {folder_path}")
            
    # Lưu toàn bộ dữ liệu ra file CSV
    if len(dataset) > 0:
        # Số lượng đặc trưng HOG sinh ra rất nhiều, ta đánh số tự động cho các cột
        num_features = len(dataset[0]) - 1
        col_names = [f'Feature_{i}' for i in range(num_features)] + ['Label']
        
        df = pd.DataFrame(dataset, columns=col_names)
        csv_path = '../data/dog_cat_features.csv'
        df.to_csv(csv_path, index=False)
        print(f"\n✅ Tuyệt vời! Đã xuất dữ liệu thành công ra file: {csv_path}")
    else:
        print("\n❌ Không có dữ liệu nào được trích xuất. Hãy kiểm tra lại ảnh trong thư mục train!")

# Gọi hàm thực thi
Create_DogCat_Dataset()

Đã xử lý xong: ../data/train\dog\dog_0.jpg
Đã xử lý xong: ../data/train\dog\dog_1.jpg
Đã xử lý xong: ../data/train\dog\dog_10.jpg
Đã xử lý xong: ../data/train\dog\dog_100.jpg
Đã xử lý xong: ../data/train\dog\dog_101.jpg
Đã xử lý xong: ../data/train\dog\dog_102.jpg
Đã xử lý xong: ../data/train\dog\dog_103.jpg
Đã xử lý xong: ../data/train\dog\dog_104.jpg
Đã xử lý xong: ../data/train\dog\dog_105.jpg
Đã xử lý xong: ../data/train\dog\dog_106.jpg
Đã xử lý xong: ../data/train\dog\dog_107.jpg
Đã xử lý xong: ../data/train\dog\dog_108.jpg
Đã xử lý xong: ../data/train\dog\dog_109.jpg
Đã xử lý xong: ../data/train\dog\dog_11.jpg
Đã xử lý xong: ../data/train\dog\dog_110.jpg
Đã xử lý xong: ../data/train\dog\dog_111.jpg
Đã xử lý xong: ../data/train\dog\dog_112.jpg
Đã xử lý xong: ../data/train\dog\dog_113.jpg
Đã xử lý xong: ../data/train\dog\dog_114.jpg
Đã xử lý xong: ../data/train\dog\dog_115.jpg
Đã xử lý xong: ../data/train\dog\dog_116.jpg
Đã xử lý xong: ../data/train\dog\dog_117.jpg
Đã xử lý xong: .